# 02 · EDA — crashes with victims

When, where and who. Figures go to `paper/figures/` (PT and EN); aggregates for the site go to `site/public/data/`.

In [1]:
# Setup: works on Colab (Drive + private repo) and locally (repo checkout).
import sys, os
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive, userdata
    drive.mount("/content/drive")
    os.environ["BHTRAFFIC_DATA_ROOT"] = "/content/drive/MyDrive/transito_bh"
    token = userdata.get("GH_TOKEN")  # Colab Secrets: fine-grained token, read-only on the repo
    # Clone (not pip-from-git): the package resolves data/ and paper/ relative to the repo root.
    url = f"https://{token}@github.com/Dias0202/Transito_bh.git"
    if os.path.isdir("/content/Transito_bh"):
        !git -C /content/Transito_bh pull -q {url} main
    else:
        !git clone -q {url} /content/Transito_bh
        !git -C /content/Transito_bh remote set-url origin https://github.com/Dias0202/Transito_bh.git
    %pip -q install -e "/content/Transito_bh[geo,viz]"
    sys.path.insert(0, "/content/Transito_bh/src")
    os.chdir("/content/Transito_bh/notebooks")
else:
    from pathlib import Path
    sys.path.insert(0, str(Path.cwd().parent / "src"))

In [2]:
import pandas as pd
from bhtraffic.config import processed_dir
d = processed_dir('sinistros')
o = pd.read_parquet(d / 'ocorrencias.parquet')
v = pd.read_parquet(d / 'veiculos.parquet')
p = pd.read_parquet(d / 'pessoas.parquet')
RECENT = (2020, 2024)
o.shape, v.shape, p.shape

((184350, 17), (333111, 5), (404957, 9))

## The decade

In [3]:
deaths = p[p.fatal].merge(o[['boletim', 'year']], on='boletim').groupby('year').size()
pd.DataFrame({'crashes': o.groupby('year').size(), 'deaths': deaths})

,crashes,deaths
year,,
2011,16288,223
2012,15140,179
2013,14077,166
2014,14907,177
2015,13178,150
2016,12473,135
2017,12243,121
2018,11656,113
2019,13342,105


## When — weekday x hour (recent window, hour informed)

In [4]:
r = o[o.year.between(*RECENT) & o.hour.notna()]
m = r.groupby(['weekday', 'hour']).size().unstack(fill_value=0)
m.style.background_gradient(cmap='Oranges', axis=None)

hour,0.000000,1.000000,2.000000,3.000000,4.000000,5.000000,6.000000,7.000000,8.000000,9.000000,10.000000,11.000000,12.000000,13.000000,14.000000,15.000000,16.000000,17.000000,18.000000,19.000000,20.000000,21.000000,22.000000,23.000000
weekday,,,,,,,,,,,,,,,,,,,,,,,,
0,166,72,50,37,43,95,339,709,612,429,409,450,476,517,495,510,580,712,698,533,373,260,227,115
1,68,39,27,25,34,74,292,656,669,454,440,457,480,496,495,537,597,722,761,595,378,282,233,140
2,116,45,36,32,29,72,277,641,665,479,422,443,496,475,509,542,574,732,786,616,375,313,248,161
3,111,60,33,27,28,76,289,617,629,459,440,434,474,472,525,534,572,736,733,639,413,327,266,189
4,137,78,53,43,49,87,294,639,592,455,420,480,554,523,573,625,751,754,732,657,551,418,337,239
5,178,140,110,99,106,130,200,297,330,364,424,516,596,584,553,518,496,427,464,490,497,415,327,270
6,205,185,144,107,135,120,157,170,149,173,213,328,363,354,362,366,368,355,410,486,426,356,284,233


## Where — districts and worst H3 cells

In [5]:
rr = o[o.year.between(*RECENT)]
rr.groupby('regional').agg(crashes=('boletim', 'size'), fatal=('fatal', 'sum')).sort_values('crashes', ascending=False)

,crashes,fatal
regional,,
CENTRO-SUL,9014,58
PAMPULHA,8699,107
OESTE,7240,84
NOROESTE,6802,73
NORDESTE,6175,85
BARREIRO,5408,51
VENDA NOVA,5096,37
LESTE,4880,36
NORTE,4852,43


In [6]:
top = rr.dropna(subset=['h3']).groupby('h3').agg(crashes=('boletim', 'size'), fatal=('fatal', 'sum'), regional=('regional', lambda s: s.mode().iat[0] if s.notna().any() else None), lat=('lat', 'median'), lon=('lon', 'median'))
top.sort_values('crashes', ascending=False).head(15)

,crashes,fatal,regional,lat,lon
h3,,,,,
88a88cdb31fffff,889,5,CENTRO-SUL,-19.914465,-43.940025
88a88cdb39fffff,853,6,CENTRO-SUL,-19.920933,-43.942721
88a88cdb3dfffff,647,5,CENTRO-SUL,-19.919710,-43.935689
88a88cdb07fffff,572,3,CENTRO-SUL,-19.926934,-43.935295
88a88cdb3bfffff,493,7,CENTRO-SUL,-19.916767,-43.948528
88a88cdb0dfffff,464,4,CENTRO-SUL,-19.941312,-43.934064
88a88cdb1dfffff,457,4,OESTE,-19.930365,-43.956776
88a88cd915fffff,440,5,NORDESTE,-19.867064,-43.928210
88a88cdb17fffff,438,2,NOROESTE,-19.917931,-43.956244


## Who — vehicles involved and deaths by road user

In [7]:
recent = set(rr.boletim)
print('share of crashes with a motorcycle:', round(rr.boletim.isin(set(v.loc[v.grupo == 'motorcycle', 'boletim'])).mean(), 3))
dead = p[p.fatal & p.boletim.isin(recent)]
dead.grupo_veiculo.where(~dead.pedestre.fillna(False).astype(bool), 'pedestrian').value_counts()

share of crashes with a motorcycle: 0.753


grupo_veiculo
motorcycle    307
pedestrian    205
car            83
bicycle        21
truck          12
bus             9
other           1
Name: count, dtype: int64

## Outputs: paper figures and site data

In [8]:
from bhtraffic.viz import figures
[f.name for f in figures.all_figures(o, p, RECENT)]

['yearly_pt.png',
 'clock_pt.png',
 'road_users_pt.png',
 'regionals_pt.png',
 'yearly_en.png',
 'clock_en.png',
 'road_users_en.png',
 'regionals_en.png']

In [9]:
from bhtraffic.export import site
site.main()

site\public\data\summary.json  0 KB
site\public\data\yearly.json  1 KB
site\public\data\clock.json  1 KB
site\public\data\hexes.json  121 KB
site\public\data\regionals.json  1 KB
site\public\data\vehicles.json  0 KB
site\public\data\bus.json  66 KB


0